# 🎯 Advanced RFM + K-means Clustering Analysis
## Combining Behavioral Segmentation with Customer Value Metrics

This notebook performs a comprehensive analysis combining:
- **K-means Clustering** (from basket characteristics)
- **RFM Segmentation** (Recency, Frequency, Monetary)
- **Customer Lifetime Value (CLV) Estimation**
- **Actionable Business Rules** for marketing strategies

---

### 📚 Outline:
1. **RFM Analysis Foundation** - Calculate R, F, M metrics
2. **RFM Scoring & Segmentation** - Quintile-based scores (1-5)
3. **Cross-tabulation: RFM vs Cluster** - Relationship mapping
4. **Customer Lifetime Value (CLV) Estimation** - Revenue forecasting
5. **Actionable Business Rules & Recommendations** - Data-driven strategies
6. **Export Results & Summary Dashboard** - Final deliverables

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully")
print(f"Total baskets in analysis: {len(final_basket_df)}")
print(f"Clusters identified: {final_basket_df['Cluster'].nunique()}")

---
## 📊 SECTION 1: RFM ANALYSIS FOUNDATION
**Objective:** Calculate Recency, Frequency, and Monetary metrics for each basket/customer segment

---

In [ ]:
# ============================================================================
# 1️⃣ PREPARE TRANSACTION DATA FOR RFM
# ============================================================================

print("="*80)
print("SECTION 1: RFM ANALYSIS FOUNDATION")
print("="*80)

# Add Cluster info back to df_pos
df_pos_with_cluster = df_pos.merge(
    final_basket_df[["Cluster"]], 
    left_on="Basket_ID", 
    right_index=True, 
    how="left"
)

print(f"\n📋 Transaction data shape: {df_pos_with_cluster.shape}")
print(f"Date range: {df_pos_with_cluster['Date'].min()} to {df_pos_with_cluster['Date'].max()}")

# Ensure Date is datetime
df_pos_with_cluster['Date'] = pd.to_datetime(df_pos_with_cluster['Date'])

# Reference date for RFM (latest date in dataset)
reference_date = df_pos_with_cluster['Date'].max()
print(f"📅 Reference date for RFM: {reference_date}")

In [ ]:
# ============================================================================
# 2️⃣ CALCULATE RFM METRICS - BASKET LEVEL
# ============================================================================

print("\n" + "="*80)
print("Calculating RFM Metrics (Basket-Level Aggregation)")
print("="*80)

# A) RECENCY - Days since last purchase (per basket from final_basket_df perspective)
# We'll compute at transaction level then aggregate
basket_dates = df_pos_with_cluster.groupby("Basket_ID").agg({
    'Date': ['min', 'max', 'count'],  # First, Last, Count of transactions
    'Value': 'sum',
    'Quantity': 'sum',
    'Cluster': 'first'
})
basket_dates.columns = ['Date_First', 'Date_Last', 'Num_Items', 'Total_Value', 'Total_Qty', 'Cluster']
basket_dates['Recency'] = (reference_date - basket_dates['Date_Last']).dt.days

# B) FREQUENCY - Number of unique transaction dates (visits) per basket_id
# Note: Since each row is a transaction, count rows per basket
frequency_data = df_pos_with_cluster.groupby("Basket_ID").size().reset_index(name='Frequency')

# C) MONETARY - Total spending per basket (already have)
monetary_data = df_pos_with_cluster.groupby("Basket_ID").agg({'Value': 'sum'}).reset_index()
monetary_data.columns = ['Basket_ID', 'Monetary']

# Merge into RFM dataframe
rfm = basket_dates.reset_index()
rfm = rfm.merge(frequency_data, on="Basket_ID", how="left")
rfm = rfm.merge(monetary_data, on="Basket_ID", how="left")

# Recalculate Frequency to avoid duplication
rfm['Frequency'] = rfm['Num_Items']

# Select final RFM columns
rfm = rfm[['Basket_ID', 'Date_First', 'Date_Last', 'Recency', 'Frequency', 'Monetary', 'Cluster']]

print(f"\n✅ RFM Dataframe created with {len(rfm)} baskets")
print(f"\nRFM Statistics:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

# Save for later
rfm_original = rfm.copy()

In [ ]:
# ============================================================================
# 3️⃣ CALCULATE RFM METRICS - CUSTOMER/CLUSTER LEVEL
# ============================================================================

print("\n" + "="*80)
print("Aggregating RFM Metrics to Cluster Level")
print("="*80)

# For RFM, we can also work at Cluster level
# This will help us understand cluster behaviors

rfm_by_cluster = rfm.groupby('Cluster').agg({
    'Recency': ['mean', 'median', 'std'],
    'Frequency': ['mean', 'median', 'std'],
    'Monetary': ['mean', 'median', 'std']
}).round(2)

print("\n📊 RFM Metrics by Cluster:")
print(rfm_by_cluster)

# Also compute some aggregate stats
rfm_cluster_summary = rfm.groupby('Cluster').agg(
    Total_Baskets=('Basket_ID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum'),
).round(2)

print("\n💰 Cluster Revenue Summary:")
print(rfm_cluster_summary)

---
## 🏆 SECTION 2: RFM SCORING & SEGMENTATION
**Objective:** Create quintile-based scores (1-5) for R, F, M and define customer segments

---

In [ ]:
# ============================================================================
# 4️⃣ CREATE QUINTILE-BASED RFM SCORES
# ============================================================================

print("="*80)
print("SECTION 2: RFM SCORING & SEGMENTATION")
print("="*80)

# For Recency: Lower is better (recent = score 5, old = score 1)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['R_Score'] = rfm['R_Score'].astype(int)

# For Frequency: Higher is better (frequent = score 5, rare = score 1)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
rfm['F_Score'] = rfm['F_Score'].astype(int)

# For Monetary: Higher is better (high spend = score 5, low spend = score 1)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
rfm['M_Score'] = rfm['M_Score'].astype(int)

# Create RFM_Segment (e.g., 555, 111, etc.)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print("\n✅ RFM Scores calculated (1-5 scale)")
print(f"\nScore Distribution:")
print(f"  R_Score (Recency):  {rfm['R_Score'].value_counts().sort_index().to_dict()}")
print(f"  F_Score (Frequency): {rfm['F_Score'].value_counts().sort_index().to_dict()}")
print(f"  M_Score (Monetary):  {rfm['M_Score'].value_counts().sort_index().to_dict()}")

print(f"\nTotal unique RFM Segments: {rfm['RFM_Segment'].nunique()}")
print(f"\nTop 15 RFM Segments by count:")
print(rfm['RFM_Segment'].value_counts().head(15))

In [ ]:
# ============================================================================
# 5️⃣ DEFINE RFM SEGMENT LABELS (BUSINESS INTERPRETATION)
# ============================================================================

print("\n" + "="*80)
print("Defining RFM Segment Labels (Business Categories)")
print("="*80)

def classify_rfm_segment(row):
    """
    Classify RFM segment into business-meaningful categories
    """
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    
    # Champions: 5,5,5 or close
    if r >= 4 and f >= 4 and m >= 4:
        return "🏆 Champions"
    # Loyal Customers: 4-5 on F and M, recent
    elif r >= 4 and f >= 4 and m >= 3:
        return "❤️ Loyal Customers"
    # Potential Loyalists: Recent, Good frequency/spend but not yet champion
    elif r >= 4 and f >= 3 and m >= 3:
        return "🌟 Potential Loyalists"
    # Big Spenders: High M, but maybe not frequent or recent
    elif m >= 5:
        return "💰 Big Spenders"
    # At-Risk (Good past value, but not recent)
    elif r <= 2 and f >= 3 and m >= 3:
        return "⚠️ At-Risk"
    # Lost: Low recency, low frequency, low monetary
    elif r <= 2 and f <= 2 and m <= 2:
        return "❌ Lost"
    # New Customers: High recency, low frequency
    elif r >= 4 and f <= 2:
        return "🆕 New Customers"
    # Cannot Lose Them: Low recency but high M or F
    elif r <= 2 and (f >= 4 or m >= 4):
        return "🚨 Cannot Lose Them"
    # Default: Needs Attention
    else:
        return "📌 Needs Attention"

rfm['RFM_Label'] = rfm.apply(classify_rfm_segment, axis=1)

print("\n✅ RFM Segment Labels assigned")
print(f"\nRFM Label Distribution:")
rfm_label_counts = rfm['RFM_Label'].value_counts()
print(rfm_label_counts)

# Calculate percentage
print(f"\nRFM Label Percentage:")
print((rfm_label_counts / len(rfm) * 100).round(1))

In [ ]:
# ============================================================================
# 6️⃣ VISUALIZE RFM SCORE DISTRIBUTIONS
# ============================================================================

print("\n" + "="*80)
print("Visualizing RFM Score Distributions")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# R_Score distribution
axes[0, 0].hist(rfm['R_Score'], bins=5, color='#FF6B6B', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Recency Score Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('R_Score (5=Recent, 1=Old)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, alpha=0.3)

# F_Score distribution
axes[0, 1].hist(rfm['F_Score'], bins=5, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Frequency Score Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('F_Score (5=Frequent, 1=Rare)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, alpha=0.3)

# M_Score distribution
axes[1, 0].hist(rfm['M_Score'], bins=5, color='#45B7D1', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Monetary Score Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('M_Score (5=High Spend, 1=Low Spend)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(True, alpha=0.3)

# RFM_Label distribution
label_counts = rfm['RFM_Label'].value_counts()
axes[1, 1].barh(range(len(label_counts)), label_counts.values, color='#95E1D3', edgecolor='black')
axes[1, 1].set_yticks(range(len(label_counts)))
axes[1, 1].set_yticklabels(label_counts.index)
axes[1, 1].set_title('RFM Segment Label Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Count')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n✅ Score distributions visualized")

In [ ]:
# ============================================================================
# 7️⃣ RFM SCORE HEATMAP (Correlation between R, F, M)
# ============================================================================

print("\n" + "="*80)
print("Creating RFM Score Correlation Heatmap")
print("="*80)

fig, ax = plt.subplots(figsize=(10, 8))

# Create heatmap of average Monetary by R_Score and F_Score
rfm_pivot = rfm.pivot_table(
    values='Monetary', 
    index='R_Score', 
    columns='F_Score', 
    aggfunc='mean'
)

sns.heatmap(rfm_pivot, annot=True, fmt='.2f', cmap='RdYlGn', 
            cbar_kws={'label': 'Avg Monetary Value (€)'}, ax=ax,
            linewidths=1, linecolor='black')
ax.set_title('Average Monetary Value by Recency & Frequency Scores', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Frequency Score (F)', fontsize=12)
ax.set_ylabel('Recency Score (R)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n✅ Heatmap created - Shows relationship between R, F, and M scores")

---
## 🔗 SECTION 3: CROSS-TABULATION - RFM vs CLUSTER
**Objective:** Map relationship between K-means clusters and RFM segments

---

In [ ]:
# ============================================================================
# 8️⃣ CROSS-TABULATION: RFM_LABEL vs CLUSTER
# ============================================================================

print("="*80)
print("SECTION 3: CROSS-TABULATION - RFM VS CLUSTER")
print("="*80)

# Create cross-tabulation
crosstab_rfm_cluster = pd.crosstab(
    rfm['Cluster'], 
    rfm['RFM_Label'], 
    margins=True
)

print("\n📊 Cross-tabulation: Cluster vs RFM Label")
print(crosstab_rfm_cluster)

# Percentage version
crosstab_pct = pd.crosstab(
    rfm['Cluster'], 
    rfm['RFM_Label'], 
    normalize='index'
) * 100

print("\n📈 Percentage Distribution (% within each Cluster):")
print(crosstab_pct.round(1))

In [ ]:
# ============================================================================
# 9️⃣ HEATMAP: CLUSTER vs RFM_LABEL
# ============================================================================

print("\n" + "="*80)
print("Visualizing Cluster-RFM Relationship")
print("="*80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap 1: Count
crosstab_count = pd.crosstab(rfm['Cluster'], rfm['RFM_Label'])
sns.heatmap(crosstab_count, annot=True, fmt='d', cmap='Blues', ax=ax1, 
            cbar_kws={'label': 'Count'}, linewidths=1, linecolor='white')
ax1.set_title('Cluster vs RFM Label (Count)', fontsize=14, fontweight='bold')
ax1.set_xlabel('RFM Label', fontsize=12)
ax1.set_ylabel('Cluster', fontsize=12)

# Heatmap 2: Percentage
sns.heatmap(crosstab_pct, annot=True, fmt='.1f', cmap='YlGn', ax=ax2,
            cbar_kws={'label': 'Percentage (%)'}, linewidths=1, linecolor='white')
ax2.set_title('Cluster vs RFM Label (% Distribution)', fontsize=14, fontweight='bold')
ax2.set_xlabel('RFM Label', fontsize=12)
ax2.set_ylabel('Cluster', fontsize=12)

plt.tight_layout()
plt.show()

print("\n✅ Heatmaps created")

In [ ]:
# ============================================================================
# 🔟 DETAILED CLUSTER-RFM PROFILES
# ============================================================================

print("\n" + "="*80)
print("Detailed Cluster-RFM Profiles")
print("="*80)

for cluster in sorted(rfm['Cluster'].unique()):
    print(f"\n{'='*80}")
    print(f"🔵 CLUSTER {cluster}")
    print(f"{'='*80}")
    
    cluster_data = rfm[rfm['Cluster'] == cluster]
    
    print(f"\nBasket Count: {len(cluster_data)} ({len(cluster_data)/len(rfm)*100:.1f}% of total)")
    
    # Top RFM Labels in this cluster
    print(f"\nTop RFM Labels:")
    top_labels = cluster_data['RFM_Label'].value_counts().head(5)
    for label, count in top_labels.items():
        print(f"  • {label:30s}: {count:6d} ({count/len(cluster_data)*100:5.1f}%)")
    
    # Average RFM metrics
    print(f"\nAverage RFM Metrics:")
    print(f"  • Avg Recency (days):   {cluster_data['Recency'].mean():.1f}")
    print(f"  • Avg Frequency:        {cluster_data['Frequency'].mean():.1f}")
    print(f"  • Avg Monetary (€):     {cluster_data['Monetary'].mean():.2f}")
    
    # Average RFM Scores
    print(f"\nAverage RFM Scores (1-5):")
    print(f"  • Avg R_Score:          {cluster_data['R_Score'].mean():.2f}")
    print(f"  • Avg F_Score:          {cluster_data['F_Score'].mean():.2f}")
    print(f"  • Avg M_Score:          {cluster_data['M_Score'].mean():.2f}")

---
## 💰 SECTION 4: CUSTOMER LIFETIME VALUE (CLV) ESTIMATION
**Objective:** Estimate CLV using RFM metrics and historical data

---

In [ ]:
# ============================================================================
# 1️⃣1️⃣ CALCULATE CUSTOMER LIFETIME VALUE (CLV)
# ============================================================================

print("="*80)
print("SECTION 4: CUSTOMER LIFETIME VALUE (CLV) ESTIMATION")
print("="*80)

# CLV can be estimated using several methods:
# Method 1: Simple CLV = Monetary * (Frequency / Recency) * Customer_Lifespan
# Method 2: CLV = Average Order Value * Purchase Frequency * Customer Lifespan
# Method 3: CLV using RFM scores as weights

# Let's use a combination approach

# A) Average Order Value (AOV)
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

# B) Purchase Frequency Rate (transactions per day)
rfm['Frequency_Rate'] = rfm['Frequency'] / (rfm['Date_Last'] - rfm['Date_First']).dt.days.clip(lower=1)

# C) Estimated Customer Lifespan (in years) - we'll assume 2 years based on cohort analysis
# More sophisticated: use RFM to predict lifespan
customer_lifespan_years = 2  # Conservative estimate

# D) CLV Calculation (simplified)
# CLV = AOV * Frequency_Rate * 365 * customer_lifespan_years
# But we need to adjust for churn probability based on Recency

# Define churn probability based on Recency
def estimate_churn_probability(recency_days, max_recency=365):
    """
    Estimate churn probability: recent customers have lower churn
    """
    return min(recency_days / max_recency, 1.0)

rfm['Churn_Probability'] = rfm['Recency'].apply(estimate_churn_probability)
rfm['Retention_Probability'] = 1 - rfm['Churn_Probability']

# E) Adjusted CLV
rfm['CLV_Estimate'] = (
    rfm['AOV'] * 
    rfm['Frequency_Rate'] * 
    365 * 
    customer_lifespan_years * 
    rfm['Retention_Probability']  # Adjust for churn
)

# Cap unrealistic values (where frequency_rate is very high or AOV is very high)
rfm['CLV_Estimate'] = rfm['CLV_Estimate'].clip(upper=rfm['CLV_Estimate'].quantile(0.99))

print("\n✅ CLV Estimated for all baskets")
print(f"\nCLV Statistics:")
print(rfm[['AOV', 'Frequency_Rate', 'Retention_Probability', 'CLV_Estimate']].describe())

print(f"\nCLV Range:")
print(f"  Min: €{rfm['CLV_Estimate'].min():.2f}")
print(f"  Max: €{rfm['CLV_Estimate'].max():.2f}")
print(f"  Mean: €{rfm['CLV_Estimate'].mean():.2f}")
print(f"  Median: €{rfm['CLV_Estimate'].median():.2f}")

In [ ]:
# ============================================================================
# 1️⃣2️⃣ CLV BY RFM SEGMENT
# ============================================================================

print("\n" + "="*80)
print("CLV Analysis by RFM Segment")
print("="*80)

clv_by_rfm_label = rfm.groupby('RFM_Label').agg(
    Basket_Count=('Basket_ID', 'count'),
    Avg_CLV=('CLV_Estimate', 'mean'),
    Median_CLV=('CLV_Estimate', 'median'),
    Total_CLV_Potential=('CLV_Estimate', 'sum'),
    Avg_Monetary=('Monetary', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
).round(2)

clv_by_rfm_label = clv_by_rfm_label.sort_values('Total_CLV_Potential', ascending=False)

print("\n💰 CLV Metrics by RFM Label:")
print(clv_by_rfm_label)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Avg CLV by RFM Label
ax1 = axes[0, 0]
clv_by_rfm_label_sorted = clv_by_rfm_label.sort_values('Avg_CLV', ascending=True)
ax1.barh(range(len(clv_by_rfm_label_sorted)), clv_by_rfm_label_sorted['Avg_CLV'].values, color='#2ECC71')
ax1.set_yticks(range(len(clv_by_rfm_label_sorted)))
ax1.set_yticklabels(clv_by_rfm_label_sorted.index)
ax1.set_xlabel('Average CLV (€)', fontsize=11)
ax1.set_title('Average CLV by RFM Label', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# 2) Total CLV Potential by RFM Label
ax2 = axes[0, 1]
clv_by_rfm_label_sorted2 = clv_by_rfm_label.sort_values('Total_CLV_Potential', ascending=True)
ax2.barh(range(len(clv_by_rfm_label_sorted2)), clv_by_rfm_label_sorted2['Total_CLV_Potential'].values, color='#3498DB')
ax2.set_yticks(range(len(clv_by_rfm_label_sorted2)))
ax2.set_yticklabels(clv_by_rfm_label_sorted2.index)
ax2.set_xlabel('Total CLV Potential (€)', fontsize=11)
ax2.set_title('Total CLV Potential by RFM Label', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# 3) Basket count by RFM Label
ax3 = axes[1, 0]
clv_by_rfm_label_sorted3 = clv_by_rfm_label.sort_values('Basket_Count', ascending=True)
ax3.barh(range(len(clv_by_rfm_label_sorted3)), clv_by_rfm_label_sorted3['Basket_Count'].values, color='#E74C3C')
ax3.set_yticks(range(len(clv_by_rfm_label_sorted3)))
ax3.set_yticklabels(clv_by_rfm_label_sorted3.index)
ax3.set_xlabel('Basket Count', fontsize=11)
ax3.set_title('Basket Count by RFM Label', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# 4) Scatter: Frequency vs CLV (colored by RFM Label)
ax4 = axes[1, 1]
for label in rfm['RFM_Label'].unique():
    label_data = rfm[rfm['RFM_Label'] == label]
    ax4.scatter(label_data['Frequency'], label_data['CLV_Estimate'], 
               alpha=0.5, label=label, s=30)
ax4.set_xlabel('Frequency', fontsize=11)
ax4.set_ylabel('CLV Estimate (€)', fontsize=11)
ax4.set_title('Frequency vs CLV by RFM Label', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9, loc='best')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ CLV visualizations created")

In [ ]:
# ============================================================================
# 1️⃣3️⃣ CLV BY CLUSTER
# ============================================================================

print("\n" + "="*80)
print("CLV Analysis by K-means Cluster")
print("="*80)

clv_by_cluster = rfm.groupby('Cluster').agg(
    Basket_Count=('Basket_ID', 'count'),
    Avg_CLV=('CLV_Estimate', 'mean'),
    Median_CLV=('CLV_Estimate', 'median'),
    Std_CLV=('CLV_Estimate', 'std'),
    Total_CLV_Potential=('CLV_Estimate', 'sum'),
    Avg_Monetary=('Monetary', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Recency=('Recency', 'mean'),
).round(2)

clv_by_cluster = clv_by_cluster.sort_values('Avg_CLV', ascending=False)

print("\n💰 CLV Metrics by Cluster:")
print(clv_by_cluster)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Avg CLV by Cluster
ax1 = axes[0, 0]
clusters = sorted(rfm['Cluster'].unique())
avg_clv_vals = [clv_by_cluster.loc[c, 'Avg_CLV'] for c in clusters]
bars1 = ax1.bar(clusters, avg_clv_vals, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters)], edgecolor='black')
ax1.set_xlabel('Cluster', fontsize=11)
ax1.set_ylabel('Average CLV (€)', fontsize=11)
ax1.set_title('Average CLV by Cluster', fontsize=12, fontweight='bold')
ax1.set_xticks(clusters)
ax1.grid(True, alpha=0.3, axis='y')
# Add value labels on bars
for i, bar in enumerate(bars1):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'€{height:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2) Total CLV Potential by Cluster
ax2 = axes[0, 1]
total_clv_vals = [clv_by_cluster.loc[c, 'Total_CLV_Potential'] for c in clusters]
bars2 = ax2.bar(clusters, total_clv_vals, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters)], edgecolor='black')
ax2.set_xlabel('Cluster', fontsize=11)
ax2.set_ylabel('Total CLV Potential (€)', fontsize=11)
ax2.set_title('Total CLV Potential by Cluster', fontsize=12, fontweight='bold')
ax2.set_xticks(clusters)
ax2.grid(True, alpha=0.3, axis='y')
for i, bar in enumerate(bars2):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'€{height:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3) Box plot: CLV distribution by Cluster
ax3 = axes[1, 0]
cluster_clv_data = [rfm[rfm['Cluster'] == c]['CLV_Estimate'].values for c in clusters]
bp = ax3.boxplot(cluster_clv_data, labels=clusters, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_xlabel('Cluster', fontsize=11)
ax3.set_ylabel('CLV Estimate (€)', fontsize=11)
ax3.set_title('CLV Distribution by Cluster', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# 4) Basket count by Cluster
ax4 = axes[1, 1]
basket_counts = [clv_by_cluster.loc[c, 'Basket_Count'] for c in clusters]
bars4 = ax4.bar(clusters, basket_counts, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters)], edgecolor='black')
ax4.set_xlabel('Cluster', fontsize=11)
ax4.set_ylabel('Basket Count', fontsize=11)
ax4.set_title('Basket Count by Cluster', fontsize=12, fontweight='bold')
ax4.set_xticks(clusters)
ax4.grid(True, alpha=0.3, axis='y')
for bar in bars4:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Cluster CLV visualizations created")

---
## 🎯 SECTION 5: ACTIONABLE BUSINESS RULES & RECOMMENDATIONS
**Objective:** Define data-driven marketing strategies for each cluster-RFM combination

---

In [ ]:
# ============================================================================
# 1️⃣4️⃣ DEFINE BUSINESS RULES & RECOMMENDATIONS
# ============================================================================

print("="*80)
print("SECTION 5: ACTIONABLE BUSINESS RULES & RECOMMENDATIONS")
print("="*80)

# Create a comprehensive recommendations matrix
recommendations = {}

for cluster in sorted(rfm['Cluster'].unique()):
    recommendations[cluster] = {}
    
    cluster_data = rfm[rfm['Cluster'] == cluster]
    
    for rfm_label in cluster_data['RFM_Label'].unique():
        segment_data = cluster_data[cluster_data['RFM_Label'] == rfm_label]
        
        if len(segment_data) == 0:
            continue
        
        # Calculate key metrics
        avg_clv = segment_data['CLV_Estimate'].mean()
        avg_recency = segment_data['Recency'].mean()
        avg_freq = segment_data['Frequency'].mean()
        basket_count = len(segment_data)
        
        # Define strategy based on CLV and RFM metrics
        if '🏆 Champions' in rfm_label:
            strategy = {
                'Priority': '🔴 CRITICAL',
                'Segment_Type': 'High-Value Retention',
                'Key_Actions': [
                    '✓ Premium loyalty program with exclusive benefits',
                    '✓ VIP customer service and priority support',
                    '✓ Early access to new products and special offers',
                    '✓ Personalized recommendations based on purchase history'
                ],
                'Pricing_Strategy': 'Premium pricing with tiered discounts',
                'Promotion_Focus': 'Cross-sell high-margin products',
                'Frequency': 'Monthly high-touch engagement'
            }
        
        elif '❤️ Loyal Customers' in rfm_label:
            strategy = {
                'Priority': '🟠 HIGH',
                'Segment_Type': 'Core Base Protection',
                'Key_Actions': [
                    '✓ Maintain consistent engagement through email/SMS',
                    '✓ Reward repeat purchases with loyalty points',
                    '✓ Exclusive member-only promotions',
                    '✓ Upsell higher-value product categories'
                ],
                'Pricing_Strategy': 'Volume discounts and bundle offers',
                'Promotion_Focus': 'Category expansion and cross-selling',
                'Frequency': 'Bi-weekly engagement'
            }
        
        elif '🌟 Potential Loyalists' in rfm_label:
            strategy = {
                'Priority': '🟡 MEDIUM-HIGH',
                'Segment_Type': 'Conversion Focus',
                'Key_Actions': [
                    '✓ Encourage higher purchase frequency',
                    '✓ Introduce to loyalty program',
                    '✓ Educational content about complementary products',
                    '✓ Limited-time promotions to increase AOV'
                ],
                'Pricing_Strategy': 'Attractive introductory offers',
                'Promotion_Focus': 'Product education and variety',
                'Frequency': 'Weekly touchpoints'
            }
        
        elif '💰 Big Spenders' in rfm_label:
            strategy = {
                'Priority': '🟠 HIGH',
                'Segment_Type': 'Revenue Driver',
                'Key_Actions': [
                    '✓ Re-engagement campaigns (low recency issue)',
                    '✓ Win-back offers with premium products',
                    '✓ VIP treatment to prevent churn',
                    '✓ Customized product recommendations'
                ],
                'Pricing_Strategy': 'Premium positioning with exclusive offers',
                'Promotion_Focus': 'High-margin luxury categories',
                'Frequency': 'Personalized quarterly campaigns'
            }
        
        elif '⚠️ At-Risk' in rfm_label:
            strategy = {
                'Priority': '🔴 CRITICAL',
                'Segment_Type': 'Retention/Win-back',
                'Key_Actions': [
                    '✓ Aggressive win-back campaigns',
                    '✓ Special "We miss you" offers',
                    '✓ Incentivize return purchases',
                    '✓ Exit surveys to understand churn reasons'
                ],
                'Pricing_Strategy': 'Heavy discounts and special bundles',
                'Promotion_Focus': 'Past favorite categories',
                'Frequency': 'Intensive re-engagement (daily-weekly)'
            }
        
        elif '❌ Lost' in rfm_label:
            strategy = {
                'Priority': '🟢 LOW (but monitor)',
                'Segment_Type': 'Reactivation Potential',
                'Key_Actions': [
                    '✓ Cost-effective reactivation campaigns',
                    '✓ Survey to understand reasons for lapse',
                    '✓ Seasonal re-engagement offers',
                    '✓ Gradual re-introduction to new products'
                ],
                'Pricing_Strategy': 'Deep promotional discounts',
                'Promotion_Focus': 'New/seasonal products',
                'Frequency': 'Occasional (seasonal campaigns)'
            }
        
        elif '🆕 New Customers' in rfm_label:
            strategy = {
                'Priority': '🟡 MEDIUM',
                'Segment_Type': 'Conversion to Loyalty',
                'Key_Actions': [
                    '✓ Onboarding sequence to encourage repeat purchase',
                    '✓ Welcome offers and loyalty enrollment',
                    '✓ Educational content on product benefits',
                    '✓ Frequent touchpoints to build habit'
                ],
                'Pricing_Strategy': 'Attractive introductory pricing',
                'Promotion_Focus': 'Varied categories to explore preferences',
                'Frequency': 'Weekly engagement'
            }
        
        elif '🚨 Cannot Lose Them' in rfm_label:
            strategy = {
                'Priority': '🔴 CRITICAL',
                'Segment_Type': 'High-Value Churn Prevention',
                'Key_Actions': [
                    '✓ Immediate re-engagement with premium incentives',
                    '✓ Dedicated account management',
                    '✓ Exclusive VIP program enrollment',
                    '✓ Personalized shopping experience'
                ],
                'Pricing_Strategy': 'Premium pricing with high-value perks',
                'Promotion_Focus': 'Luxury/exclusive products',
                'Frequency': 'High-touch (weekly-biweekly)'
            }
        
        else:  # 📌 Needs Attention
            strategy = {
                'Priority': '🟡 MEDIUM',
                'Segment_Type': 'Mixed Profile',
                'Key_Actions': [
                    '✓ Segmented approach based on sub-metrics',
                    '✓ A/B test different engagement strategies',
                    '✓ Monitor RFM movement for classification',
                    '✓ Flexible promotional approach'
                ],
                'Pricing_Strategy': 'Flexible, test-and-learn',
                'Promotion_Focus': 'Varied by sub-segment',
                'Frequency': 'Fortnightly touchpoints'
            }
        
        recommendations[cluster][rfm_label] = {
            'Basket_Count': basket_count,
            'Avg_CLV': avg_clv,
            'Avg_Recency_Days': avg_recency,
            'Avg_Frequency': avg_freq,
            **strategy
        }

print("\n✅ Recommendations matrix created with business strategies")

In [ ]:
# ============================================================================
# 1️⃣5️⃣ PRINT DETAILED RECOMMENDATIONS
# ============================================================================

print("\n" + "="*80)
print("DETAILED BUSINESS RECOMMENDATIONS BY CLUSTER & RFM")
print("="*80)

for cluster in sorted(recommendations.keys()):
    print(f"\n\n{'#'*80}")
    print(f"{'#'*80}")
    print(f"### CLUSTER {cluster} - DETAILED STRATEGY")
    print(f"{'#'*80}")
    print(f"{'#'*80}\n")
    
    for rfm_label in sorted(recommendations[cluster].keys()):
        rec = recommendations[cluster][rfm_label]
        
        print(f"\n┌─────────────────────────────────────────────────────────────────────────────┐")
        print(f"│ {rfm_label:77s} │")
        print(f"├─────────────────────────────────────────────────────────────────────────────┤")
        
        print(f"│ 📊 METRICS:")
        print(f"│   • Basket Count:      {rec['Basket_Count']:6d}")
        print(f"│   • Avg CLV:           €{rec['Avg_CLV']:7.2f}")
        print(f"│   • Avg Recency:       {rec['Avg_Recency_Days']:6.1f} days")
        print(f"│   • Avg Frequency:     {rec['Avg_Frequency']:6.1f}")
        
        print(f"│")
        print(f"│ 🎯 STRATEGY OVERVIEW:")
        print(f"│   Priority:            {rec['Priority']}")
        print(f"│   Segment Type:        {rec['Segment_Type']}")
        
        print(f"│")
        print(f"│ 📋 KEY ACTIONS:")
        for action in rec['Key_Actions']:
            print(f"│   {action}")
        
        print(f"│")
        print(f"│ 💳 PRICING STRATEGY:   {rec['Pricing_Strategy']}")
        print(f"│ 📢 PROMOTION FOCUS:    {rec['Promotion_Focus']}")
        print(f"│ 🔔 ENGAGEMENT FREQ:    {rec['Frequency']}")
        print(f"└─────────────────────────────────────────────────────────────────────────────┘")

In [ ]:
# ============================================================================
# 1️⃣6️⃣ SUMMARY RECOMMENDATIONS TABLE
# ============================================================================

print("\n" + "="*80)
print("EXECUTIVE SUMMARY - QUICK REFERENCE TABLE")
print("="*80)

summary_data = []

for cluster in sorted(recommendations.keys()):
    for rfm_label in sorted(recommendations[cluster].keys()):
        rec = recommendations[cluster][rfm_label]
        summary_data.append({
            'Cluster': cluster,
            'RFM_Label': rfm_label,
            'Baskets': rec['Basket_Count'],
            'Avg_CLV': f"€{rec['Avg_CLV']:.2f}",
            'Priority': rec['Priority'],
            'Segment_Type': rec['Segment_Type'],
            'Primary_Action': rec['Key_Actions'][0].replace('✓ ', '')[:40],
            'Engagement': rec['Frequency']
        })

summary_table = pd.DataFrame(summary_data)
print("\n")
print(summary_table.to_string(index=False))

# Save summary table for export
summary_recommendations_export = summary_table.copy()

---
## 📊 SECTION 6: EXPORT RESULTS & SUMMARY DASHBOARD
**Objective:** Create comprehensive dashboard and export all results

---

In [ ]:
# ============================================================================
# 1️⃣7️⃣ CREATE COMPREHENSIVE SUMMARY DASHBOARD
# ============================================================================

print("="*80)
print("SECTION 6: EXPORT RESULTS & SUMMARY DASHBOARD")
print("="*80)

fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.3)

# 1) RFM Score Distribution (Pie)
ax1 = fig.add_subplot(gs[0, 0])
rfm_label_dist = rfm['RFM_Label'].value_counts()
colors_pie = plt.cm.Set3(range(len(rfm_label_dist)))
ax1.pie(rfm_label_dist.values, labels=rfm_label_dist.index, autopct='%1.1f%%',
        colors=colors_pie, startangle=90)
ax1.set_title('RFM Segment Distribution', fontsize=12, fontweight='bold')

# 2) Cluster Distribution (Pie)
ax2 = fig.add_subplot(gs[0, 1])
cluster_dist = rfm['Cluster'].value_counts().sort_index()
colors_cluster = ['#FF6B6B', '#4ECDC4', '#45B7D1']
ax2.pie(cluster_dist.values, labels=[f'Cluster {c}' for c in cluster_dist.index],
        autopct='%1.1f%%', colors=colors_cluster, startangle=90)
ax2.set_title('Cluster Distribution', fontsize=12, fontweight='bold')

# 3) CLV by RFM Label (Top 8)
ax3 = fig.add_subplot(gs[0, 2])
clv_rfm_top = clv_by_rfm_label.nlargest(8, 'Avg_CLV').sort_values('Avg_CLV')
ax3.barh(range(len(clv_rfm_top)), clv_rfm_top['Avg_CLV'].values, color='#2ECC71')
ax3.set_yticks(range(len(clv_rfm_top)))
ax3.set_yticklabels([x[:25] for x in clv_rfm_top.index], fontsize=9)
ax3.set_xlabel('Avg CLV (€)', fontsize=10)
ax3.set_title('Top 8 RFM Segments by Avg CLV', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# 4) R Score Heatmap
ax4 = fig.add_subplot(gs[1, 0])
r_dist = rfm['R_Score'].value_counts().sort_index()
ax4.bar(r_dist.index, r_dist.values, color='#FF6B6B', edgecolor='black', alpha=0.7)
ax4.set_xlabel('Recency Score', fontsize=10)
ax4.set_ylabel('Count', fontsize=10)
ax4.set_title('Recency Score Distribution', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# 5) F Score Distribution
ax5 = fig.add_subplot(gs[1, 1])
f_dist = rfm['F_Score'].value_counts().sort_index()
ax5.bar(f_dist.index, f_dist.values, color='#4ECDC4', edgecolor='black', alpha=0.7)
ax5.set_xlabel('Frequency Score', fontsize=10)
ax5.set_ylabel('Count', fontsize=10)
ax5.set_title('Frequency Score Distribution', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='y')

# 6) M Score Distribution
ax6 = fig.add_subplot(gs[1, 2])
m_dist = rfm['M_Score'].value_counts().sort_index()
ax6.bar(m_dist.index, m_dist.values, color='#45B7D1', edgecolor='black', alpha=0.7)
ax6.set_xlabel('Monetary Score', fontsize=10)
ax6.set_ylabel('Count', fontsize=10)
ax6.set_title('Monetary Score Distribution', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

# 7) Avg CLV by Cluster
ax7 = fig.add_subplot(gs[2, 0])
clusters_sorted = sorted(rfm['Cluster'].unique())
clv_cluster_vals = [clv_by_cluster.loc[c, 'Avg_CLV'] for c in clusters_sorted]
bars = ax7.bar(clusters_sorted, clv_cluster_vals, 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters_sorted)],
               edgecolor='black', alpha=0.7)
ax7.set_xlabel('Cluster', fontsize=10)
ax7.set_ylabel('Average CLV (€)', fontsize=10)
ax7.set_title('Average CLV by Cluster', fontsize=12, fontweight='bold')
ax7.set_xticks(clusters_sorted)
ax7.grid(True, alpha=0.3, axis='y')
for bar in bars:
    height = bar.get_height()
    ax7.text(bar.get_x() + bar.get_width()/2., height,
            f'€{height:.0f}', ha='center', va='bottom', fontsize=9)

# 8) Total Baskets by Cluster
ax8 = fig.add_subplot(gs[2, 1])
basket_by_cluster = [clv_by_cluster.loc[c, 'Basket_Count'] for c in clusters_sorted]
bars2 = ax8.bar(clusters_sorted, basket_by_cluster,
                color=['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters_sorted)],
                edgecolor='black', alpha=0.7)
ax8.set_xlabel('Cluster', fontsize=10)
ax8.set_ylabel('Basket Count', fontsize=10)
ax8.set_title('Total Baskets by Cluster', fontsize=12, fontweight='bold')
ax8.set_xticks(clusters_sorted)
ax8.grid(True, alpha=0.3, axis='y')
for bar in bars2:
    height = bar.get_height()
    ax8.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}', ha='center', va='bottom', fontsize=9)

# 9) Total CLV Potential by Cluster
ax9 = fig.add_subplot(gs[2, 2])
total_clv_vals = [clv_by_cluster.loc[c, 'Total_CLV_Potential'] for c in clusters_sorted]
bars3 = ax9.bar(clusters_sorted, total_clv_vals,
                color=['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(clusters_sorted)],
                edgecolor='black', alpha=0.7)
ax9.set_xlabel('Cluster', fontsize=10)
ax9.set_ylabel('Total CLV Potential (€)', fontsize=10)
ax9.set_title('Total CLV Potential by Cluster', fontsize=12, fontweight='bold')
ax9.set_xticks(clusters_sorted)
ax9.grid(True, alpha=0.3, axis='y')
for bar in bars3:
    height = bar.get_height()
    ax9.text(bar.get_x() + bar.get_width()/2., height,
            f'€{height:,.0f}', ha='center', va='bottom', fontsize=8)

# 10) Cluster vs RFM Heatmap
ax10 = fig.add_subplot(gs[3, :])
crosstab_cluster_rfm = pd.crosstab(rfm['Cluster'], rfm['RFM_Label'])
sns.heatmap(crosstab_cluster_rfm, annot=True, fmt='d', cmap='YlOrRd', ax=ax10,
            cbar_kws={'label': 'Count'}, linewidths=1)
ax10.set_title('Cluster vs RFM Label Heatmap', fontsize=12, fontweight='bold')
ax10.set_xlabel('RFM Label', fontsize=10)
ax10.set_ylabel('Cluster', fontsize=10)

fig.suptitle('🎯 COMPREHENSIVE RFM + CLUSTERING DASHBOARD', 
             fontsize=16, fontweight='bold', y=0.995)

plt.show()

print("\n✅ Comprehensive dashboard created")

In [ ]:
# ============================================================================
# 1️⃣8️⃣ PREPARE EXPORT DATASETS
# ============================================================================

print("\n" + "="*80)
print("Preparing Export Datasets")
print("="*80)

# A) RFM Master Table (all baskets with scores and CLV)
export_rfm_master = rfm.copy()
export_rfm_master = export_rfm_master[[
    'Basket_ID', 'Date_First', 'Date_Last', 'Recency', 'Frequency', 'Monetary',
    'R_Score', 'F_Score', 'M_Score', 'RFM_Segment', 'RFM_Label',
    'AOV', 'Churn_Probability', 'Retention_Probability', 'CLV_Estimate', 'Cluster'
]].round(3)

# B) Cluster Summary
export_cluster_summary = clv_by_cluster.copy()

# C) RFM Label Summary
export_rfm_summary = clv_by_rfm_label.copy()

# D) Cluster-RFM Crosstab
export_crosstab = pd.crosstab(rfm['Cluster'], rfm['RFM_Label'], margins=True)

# E) Detailed Recommendations (flatten from nested dict)
export_recommendations = pd.DataFrame()
for cluster in sorted(recommendations.keys()):
    for rfm_label in sorted(recommendations[cluster].keys()):
        rec = recommendations[cluster][rfm_label]
        rec_row = {
            'Cluster': cluster,
            'RFM_Label': rfm_label,
            'Basket_Count': rec['Basket_Count'],
            'Avg_CLV_€': round(rec['Avg_CLV'], 2),
            'Priority': rec['Priority'],
            'Segment_Type': rec['Segment_Type'],
            'Action_1': rec['Key_Actions'][0].replace('✓ ', ''),
            'Action_2': rec['Key_Actions'][1].replace('✓ ', '') if len(rec['Key_Actions']) > 1 else '',
            'Action_3': rec['Key_Actions'][2].replace('✓ ', '') if len(rec['Key_Actions']) > 2 else '',
            'Pricing_Strategy': rec['Pricing_Strategy'],
            'Promotion_Focus': rec['Promotion_Focus'],
            'Engagement_Frequency': rec['Frequency']
        }
        export_recommendations = pd.concat([export_recommendations, pd.DataFrame([rec_row])], ignore_index=True)

print(f"✅ Export datasets prepared:")
print(f"   • RFM Master Table: {len(export_rfm_master)} rows")
print(f"   • Cluster Summary: {len(export_cluster_summary)} rows")
print(f"   • RFM Label Summary: {len(export_rfm_summary)} rows")
print(f"   • Cluster-RFM Crosstab: {export_crosstab.shape}")
print(f"   • Recommendations: {len(export_recommendations)} segments")